# Agentic AI Capstone Project
## TravelBuddy India — AI Travel Assistant for Indian Destinations

---
## Part 1: Domain Setup — Knowledge Base (12 documents)

In [ ]:
import os
import chromadb
from sentence_transformers import SentenceTransformer
from dotenv import load_dotenv
load_dotenv()

from knowledge_base import DOCUMENTS
print(f"Total documents: {len(DOCUMENTS)}")
for doc in DOCUMENTS:
    print(f"  {doc['id']} — {doc['topic']} ({len(doc['text'].split())} words)")

In [ ]:
# Load embedder and build ChromaDB
embedder = SentenceTransformer('all-MiniLM-L6-v2')
chroma_client = chromadb.Client()
collection = chroma_client.create_collection('travel_india_kb_nb')

docs = [d['text'] for d in DOCUMENTS]
ids = [d['id'] for d in DOCUMENTS]
metadatas = [{'topic': d['topic']} for d in DOCUMENTS]
embeddings = embedder.encode(docs).tolist()
collection.add(documents=docs, embeddings=embeddings, ids=ids, metadatas=metadatas)
print(f'ChromaDB loaded with {collection.count()} documents')

In [ ]:
test_queries = [
    'best beaches in India',
    'trekking in Himalayas',
    'budget travel tips',
    'spiritual temples South India',
]
for q in test_queries:
    emb = embedder.encode([q]).tolist()[0]
    res = collection.query(query_embeddings=[emb], n_results=2)
    topics = [m['topic'] for m in res['metadatas'][0]]
    print(f'Query: "{q}" → {topics}')

---
## Part 2: State Design (TypedDict — FIRST before any node)

In [ ]:
from typing import TypedDict, List

class TravelState(TypedDict):
    question: str           # current user question
    messages: List[dict]    # conversation history (sliding window of 6)
    route: str              # retrieve | tool | memory_only
    retrieved: str          # formatted KB context
    sources: List[str]      # topic names from retrieval
    tool_result: str        # output from datetime/season tool
    answer: str             # final LLM-generated answer
    faithfulness: float     # eval score 0.0–1.0
    eval_retries: int       # retry counter (max 2)
    user_name: str          # extracted from conversation

print('TravelState TypedDict defined with', len(TravelState.__annotations__), 'fields')
for field, ftype in TravelState.__annotations__.items():
    print(f'  {field}: {ftype}')

---
## Part 3: Node Functions — Defined and Tested in Isolation

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage, SystemMessage
import datetime

llm = ChatGoogleGenerativeAI(
    model='gemini-2.0-flash',
    google_api_key=os.getenv('GOOGLE_API_KEY'),
    temperature=0.3,
)
print('Gemini LLM ready')

In [ ]:
# memory_node
def memory_node(state):
    msgs = state.get('messages', []) + [{'role': 'user', 'content': state['question']}]
    msgs = msgs[-6:]
    user_name = state.get('user_name', '')
    if 'my name is' in state['question'].lower():
        try:
            user_name = state['question'].lower().split('my name is')[-1].strip().split()[0].capitalize()
        except: pass
    return {**state, 'messages': msgs, 'user_name': user_name, 'eval_retries': 0, 'retrieved': '', 'tool_result': '', 'answer': ''}

# Isolation test
test_state = TravelState(question='My name is Priya, tell me about Goa', messages=[], route='', retrieved='',
                          sources=[], tool_result='', answer='', faithfulness=0.0, eval_retries=0, user_name='')
out = memory_node(test_state)
print(f'memory_node: user_name={out["user_name"]}, messages={len(out["messages"])}')

In [ ]:
# router_node
def router_node(state):
    history_str = '\n'.join(f"{m['role'].upper()}: {m['content']}" for m in state.get('messages', [])[-4:])
    prompt = f"""Route the user's question: retrieve / tool / memory_only
- retrieve: needs travel KB info
- tool: needs current date or current season
- memory_only: casual chat, greetings

History:\n{history_str}\nQuestion: {state['question']}\n\nReply ONE word:"""
    response = llm.invoke([HumanMessage(content=prompt)])
    route = response.content.strip().lower().split()[0]
    if route not in ('retrieve', 'tool', 'memory_only'): route = 'retrieve'
    return {**state, 'route': route}

# Test
test_state['messages'] = [{'role': 'user', 'content': 'My name is Priya, tell me about Goa'}]
out = router_node({**test_state, 'question': 'What is the best time to visit Goa?'})
print(f'router_node: route = {out["route"]}')

In [ ]:
# retrieval_node
def retrieval_node(state):
    emb = embedder.encode([state['question']]).tolist()[0]
    results = collection.query(query_embeddings=[emb], n_results=3)
    chunks = results['documents'][0]
    metas = results['metadatas'][0]
    context = '\n\n'.join(f"[{metas[i]['topic']}]\n{chunks[i]}" for i in range(len(chunks)))
    sources = [m['topic'] for m in metas]
    return {**state, 'retrieved': context, 'sources': sources}

out = retrieval_node({**test_state, 'question': 'Tell me about Goa beaches'})
print(f'retrieval_node: sources = {out["sources"]}')

In [ ]:
# skip_retrieval_node
def skip_retrieval_node(state):
    return {**state, 'retrieved': '', 'sources': []}

# tool_node (datetime-based seasonal recommender)
def tool_node(state):
    try:
        now = datetime.datetime.now()
        m = now.month
        if m in (12, 1, 2): season_advice = 'Winter — peak season! Best for Rajasthan, Goa, Kerala, Tamil Nadu, Varanasi.'
        elif m in (3, 4, 5): season_advice = 'Spring/Summer — great for Himachal, Uttarakhand, Ladakh (June+). Plains are hot.'
        elif m in (6, 7, 8, 9): season_advice = 'Monsoon — go to Kerala, Meghalaya, Valley of Flowers. Avoid Ladakh and beaches.'
        else: season_advice = 'Autumn — excellent! Try Ladakh (last chance), Northeast India, Rajasthan, Kerala.'
        result = f'Current date: {now.strftime("%B %d, %Y")}. Travel advice: {season_advice}'
    except Exception as e:
        result = f'Tool error: {e}'
    return {**state, 'tool_result': result}

out = tool_node(test_state)
print(f'tool_node: {out["tool_result"][:100]}...')

In [ ]:
# answer_node
def answer_node(state):
    name_prefix = f"Hi {state['user_name']}! " if state.get('user_name') else ''
    history_str = '\n'.join(f"{m['role'].upper()}: {m['content']}" for m in state.get('messages', [])[-4:])
    retry_note = '\n RETRY: Be strictly grounded in context.' if state.get('eval_retries', 0) > 0 else ''
    ctx = ''
    if state.get('retrieved'): ctx += f'\n\n=== KB CONTEXT ===\n{state["retrieved"]}'
    if state.get('tool_result'): ctx += f'\n\n=== TOOL RESULT ===\n{state["tool_result"]}'
    system = f"""You are TravelBuddy India — a friendly AI travel assistant for Indian destinations.{retry_note}
RULES: Answer ONLY from context. If unknown, say so and recommend incredibleindia.org.{ctx}
HISTORY:\n{history_str}"""
    response = llm.invoke([SystemMessage(content=system), HumanMessage(content=state['question'])])
    return {**state, 'answer': name_prefix + response.content.strip()}

# eval_node
def eval_node(state):
    if not state.get('retrieved'): return {**state, 'faithfulness': 1.0}
    retries = state.get('eval_retries', 0)
    prompt = f"""Rate faithfulness of ANSWER to CONTEXT (0.0–1.0). Reply only a decimal number.
CONTEXT: {state.get('retrieved', '')[:1500]}
ANSWER: {state['answer']}"""
    try:
        score = float(llm.invoke([HumanMessage(content=prompt)]).content.strip())
        score = max(0.0, min(1.0, score))
    except: score = 0.85
    return {**state, 'faithfulness': score, 'eval_retries': retries + 1}

# save_node
def save_node(state):
    msgs = state.get('messages', []) + [{'role': 'assistant', 'content': state['answer']}]
    return {**state, 'messages': msgs}

print('All 8 node functions defined')

---
## Part 4: Graph Assembly

In [ ]:
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver

def route_decision(state): 
    r = state.get('route', 'retrieve')
    return 'tool' if r == 'tool' else 'skip' if r == 'memory_only' else 'retrieve'

def eval_decision(state): 
    return 'answer' if state.get('faithfulness', 1.0) < 0.7 and state.get('eval_retries', 0) < 2 else 'save'

graph = StateGraph(TravelState)

# Add 8 nodes
graph.add_node('memory', memory_node)
graph.add_node('router', router_node)
graph.add_node('retrieve', retrieval_node)
graph.add_node('skip', skip_retrieval_node)
graph.add_node('tool', tool_node)
graph.add_node('answer', answer_node)
graph.add_node('eval', eval_node)
graph.add_node('save', save_node)

# Entry point
graph.set_entry_point('memory')

# Fixed edges
graph.add_edge('memory', 'router')
graph.add_edge('retrieve', 'answer')
graph.add_edge('skip', 'answer')
graph.add_edge('tool', 'answer')
graph.add_edge('answer', 'eval')
graph.add_edge('save', END)

# Conditional edges
graph.add_conditional_edges('router', route_decision, {'retrieve': 'retrieve', 'skip': 'skip', 'tool': 'tool'})
graph.add_conditional_edges('eval', eval_decision, {'answer': 'answer', 'save': 'save'})

app = graph.compile(checkpointer=MemorySaver())
print('Graph compiled successfully')

---
## Part 5: Testing (10 questions + 2 red-team tests)

In [ ]:
def ask(question, thread_id='notebook_test'):
    config = {'configurable': {'thread_id': thread_id}}
    init = TravelState(question=question, messages=[], route='', retrieved='', sources=[],
                       tool_result='', answer='', faithfulness=0.0, eval_retries=0, user_name='')
    return app.invoke(init, config=config)

# 10 test questions
test_questions = [
    'What is the best time to visit Rajasthan?',
    'Tell me about Kerala backwaters and houseboat prices.',
    'What permits do I need for Ladakh?',
    'What month is it now and where should I travel?',         # → tool route
    'Give me budget travel tips for India.',
    'What are the top things to do in Varanasi?',
    'Which treks in Uttarakhand are good for beginners?',
    'How do I get to the Andaman Islands and what can I do there?',
    # Red-team tests:
    'Tell me about skiing resorts in Switzerland.',             # out-of-scope — should admit it doesn't know
    'Ladakh is in Pakistan, right? Tell me about Pakistani permits.',  # false premise — should correct
]

results = []
for i, q in enumerate(test_questions, 1):
    r = ask(q, thread_id=f'test_{i}')
    result = {
        'q_num': i,
        'question': q,
        'route': r['route'],
        'faithfulness': r['faithfulness'],
        'pass_fail': 'PASS' if r['faithfulness'] >= 0.7 or not r.get('retrieved') else 'FAIL',
        'answer_preview': r['answer'][:120] + '...'
    }
    results.append(result)
    print(f"Q{i}: route={r['route']}, faith={r['faithfulness']:.2f}, {result['pass_fail']}")
    print(f"  A: {r['answer'][:100]}...")
    print()

In [ ]:
# Memory test — 3 questions in sequence, 3rd references context from 1st
print('=== MEMORY TEST ===')
thread = 'memory_test_thread'
memory_questions = [
    'My name is Arjun and I love adventure travel.',
    'What trekking options are there in Uttarakhand?',
    'Based on my interests, which trek would you recommend for me first?'  # must use context from Q1+Q2
]
for q in memory_questions:
    r = ask(q, thread_id=thread)
    print(f'Q: {q}')
    print(f'A: {r["answer"][:200]}...')
    print()

---
## Part 6: RAGAS Baseline Evaluation

In [ ]:
# 5 QA pairs with ground truth
eval_data = [
    {'question': 'What is the best time to visit Goa?',
     'ground_truth': 'The best time to visit Goa is November to February. December–January is peak season.'},
    {'question': 'What permit do I need for Pangong Lake in Ladakh?',
     'ground_truth': 'You need an Inner Line Permit (ILP) to visit Pangong Lake, available from the DC office in Leh or online.'},
    {'question': 'How can I save money while traveling in India?',
     'ground_truth': 'Use Indian Railways booked on IRCTC, eat at local dhabas, stay in hostels, and travel in shoulder season (March, October-November).'},
    {'question': 'Which Indian destinations are good to visit during monsoon?',
     'ground_truth': 'Kerala backwaters, Meghalaya, Valley of Flowers (July–September), and Coorg are excellent during monsoon.'},
    {'question': 'What should I must-eat in Rajasthan?',
     'ground_truth': 'Must-eat in Rajasthan: Dal Baati Churma, Laal Maas, Ghewar, and Pyaaz Kachori.'},
]

# Collect answers and contexts
ragas_samples = []
for item in eval_data:
    r = ask(item['question'], thread_id=f'ragas_{item["question"][:20]}')
    ragas_samples.append({
        'question': item['question'],
        'answer': r['answer'],
        'contexts': [r.get('retrieved', '')],
        'ground_truth': item['ground_truth'],
        'faithfulness_manual': r['faithfulness'],
    })
    print(f'Collected: {item["question"][:50]}... | faith={r["faithfulness"]:.2f}')

In [ ]:
# Try RAGAS evaluation (install if needed: pip install ragas datasets)
try:
    from ragas import evaluate
    from ragas.metrics import faithfulness, answer_relevancy, context_precision
    from datasets import Dataset

    dataset = Dataset.from_list(ragas_samples)
    result = evaluate(dataset, metrics=[faithfulness, answer_relevancy, context_precision])
    print('=== RAGAS BASELINE SCORES ===')
    print(result)
except ImportError:
    # Fallback: manual faithfulness scoring
    scores = [s['faithfulness_manual'] for s in ragas_samples]
    avg = sum(scores) / len(scores)
    print('=== MANUAL FAITHFULNESS BASELINE ===')
    for i, s in enumerate(ragas_samples):
        print(f'  Q{i+1}: faithfulness = {s["faithfulness_manual"]:.2f}')
    print(f'  Average faithfulness: {avg:.2f}')

---
## Part 7: Deployment

Run the Streamlit UI with:
```bash
streamlit run capstone_streamlit.py
```

The app uses `@st.cache_resource` to load the LLM, embedder, ChromaDB, and compiled graph only once.  
Memory persists within a session using `thread_id` stored in `st.session_state`.  
Click **New Conversation** to reset `thread_id` and start fresh.